# 🚧 Notebook 1 — YOLOv8x-seg Pothole Detection Training

**Model:** YOLOv8x-seg (instance segmentation)  
**Output:** `yolov8x-seg-pothole.pt` → copy to `backend/models/`  
**Runtime:** GPU (T4 recommended)

## Datasets used
| Dataset | What it adds | Format in source | Used as |
|---|---|---|---|
| [PothRGBD — RGB+Depth potholes](https://www.kaggle.com/datasets/mahyeks/pothrgbd-rgb-and-depth-images-of-potholes) | Main dataset | YOLO txt labels + depth `.npy` | YOLO-seg polygons |
| [Annotated Potholes (chitholian)](https://www.kaggle.com/datasets/chitholian/annotated-potholes-dataset) | Extra annotations | Pascal VOC XML | YOLO-seg polygons |
| Your ZIP uploads (optional) | Any extra pothole datasets | Auto-detected: YOLO / VOC XML / COCO JSON | YOLO-seg polygons |

> ⚡ Set runtime to **GPU (T4)** before running.

In [ ]:
# ── 1) Check GPU ─────────────────────────────────────────────────────────────
!nvidia-smi
import torch

print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ GPU not detected. In Colab: Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# ── 2) Install dependencies ──────────────────────────────────────────────────
!pip install -q --upgrade kagglehub
!pip install -q --upgrade ultralytics opencv-python-headless albumentations pyyaml
print("✅ Dependencies installed")

In [ ]:
# ── 3) Mount Google Drive (save checkpoints) ────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/pothole_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"✅ Checkpoints will be saved to: {SAVE_DIR}")

In [ ]:
# ── 4) Kaggle authentication (no JSON upload needed) ────────────────────────
# Kaggle -> Settings -> API -> Create New Token

import os, json
from getpass import getpass

KAGGLE_USERNAME = input('Enter your Kaggle username: ').strip()
KAGGLE_KEY = getpass('Enter your Kaggle API key: ').strip()

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

print(f"✅ Kaggle credentials configured for: {KAGGLE_USERNAME}")

In [ ]:
# ── 5) Download core datasets ────────────────────────────────────────────────
import kagglehub

print('📥 Downloading PothRGBD...')
pothrgbd_path = kagglehub.dataset_download('mahyeks/pothrgbd-rgb-and-depth-images-of-potholes')
print('   ->', pothrgbd_path)

print('📥 Downloading Annotated Potholes (VOC XML)...')
annotated_path = kagglehub.dataset_download('chitholian/annotated-potholes-dataset')
print('   ->', annotated_path)

print('\n✅ Core datasets ready.')

In [ ]:
# ── 6) (Optional) Upload extra dataset ZIPs ─────────────────────────────────
# Supports annotation formats inside ZIP:
# - YOLO txt
# - Pascal VOC XML
# - COCO JSON

import zipfile
from pathlib import Path
from google.colab import files as colab_files

EXTRA_ZIP_DIR = Path('/content/extra_datasets')
EXTRA_ZIP_DIR.mkdir(parents=True, exist_ok=True)

print('📂 Upload ZIP files (Cancel if not needed):')
try:
    uploaded = colab_files.upload()
    if uploaded:
        for fname, data in uploaded.items():
            zip_dest = EXTRA_ZIP_DIR / fname
            zip_dest.write_bytes(data)
            out_dir = EXTRA_ZIP_DIR / Path(fname).stem
            out_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_dest, 'r') as zf:
                zf.extractall(out_dir)
            print(f'  ✅ Extracted {fname} -> {out_dir}')
        print(f'\n✅ Extracted {len(uploaded)} ZIP file(s) into {EXTRA_ZIP_DIR}')
    else:
        print('⏭️ No ZIP uploaded. Using core datasets only.')
except Exception as e:
    print(f'⏭️ Upload skipped ({e}). Using core datasets only.')

In [ ]:
# ── 7) Explore dataset structures ────────────────────────────────────────────
import os
from pathlib import Path

def tree(path, max_depth=3, indent=0, max_items=20):
    if indent > max_depth:
        return
    try:
        items = sorted(os.listdir(path))
    except Exception:
        return
    for item in items[:max_items]:
        full = os.path.join(path, item)
        print(' ' * indent * 2 + ('📁 ' if os.path.isdir(full) else '📄 ') + item)
        if os.path.isdir(full):
            tree(full, max_depth=max_depth, indent=indent + 1, max_items=max_items)

print('=== PothRGBD ===')
tree(pothrgbd_path)
print('\n=== Annotated Potholes ===')
tree(annotated_path)

print('\n[quick counts]')
print('PothRGBD images (.jpg/.png):', len(list(Path(pothrgbd_path).rglob('*.jpg'))) + len(list(Path(pothrgbd_path).rglob('*.png'))))
print('PothRGBD labels (.txt):      ', len(list(Path(pothrgbd_path).rglob('*.txt'))))
print('Annotated XML (.xml):        ', len(list(Path(annotated_path).rglob('*.xml'))))
print('Annotated images (.jpg/.png):', len(list(Path(annotated_path).rglob('*.jpg'))) + len(list(Path(annotated_path).rglob('*.png'))))

In [ ]:
# ── 8) Build unified YOLO-seg dataset ───────────────────────────────────────
# Source formats handled:
# - PothRGBD: YOLO txt bboxes/polygons
# - Annotated Potholes: Pascal VOC XML
# - Optional ZIP uploads: YOLO txt / VOC XML / COCO JSON

import json
import random
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

random.seed(42)

DATASET_ROOT = Path('/content/pothole_yolo_dataset')
if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)

for split in ['train', 'val']:
    (DATASET_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)

IMG_EXTS = ('.jpg', '.jpeg', '.png')

def build_image_index(base_dir: Path, skip_depth: bool = False):
    idx = {}
    for ext in IMG_EXTS:
        for p in base_dir.rglob(f'*{ext}'):
            low = str(p).lower()
            if skip_depth and ('depth' in low or p.name.lower().endswith('_depth' + ext)):
                continue
            idx.setdefault(p.stem, []).append(p)
    return idx

def bbox_to_polygon_label(xc, yc, w, h):
    x1 = xc - w / 2
    y1 = yc - h / 2
    x2 = xc + w / 2
    y2 = yc + h / 2
    pts = [x1, y1, x2, y1, x2, y2, x1, y2]
    return '0 ' + ' '.join(f'{max(0.0, min(1.0, v)):.6f}' for v in pts)

def normalize_polygon(values):
    out = []
    for i, v in enumerate(values):
        fv = float(v)
        if i % 2 == 0:  # x
            out.append(max(0.0, min(1.0, fv)))
        else:          # y
            out.append(max(0.0, min(1.0, fv)))
    return out

def parse_yolo_label_file(label_path: Path):
    lines = []
    text = label_path.read_text().strip()
    if not text:
        return lines

    for raw in text.splitlines():
        parts = raw.strip().split()
        if len(parts) < 5:
            continue

        # YOLO bbox: class xc yc w h
        if len(parts) == 5:
            try:
                xc, yc, w, h = map(float, parts[1:])
                if w > 0 and h > 0:
                    lines.append(bbox_to_polygon_label(xc, yc, w, h))
            except Exception:
                continue
        else:
            # YOLO polygon: class x1 y1 x2 y2 ...
            try:
                coords = normalize_polygon(parts[1:])
                if len(coords) >= 6 and len(coords) % 2 == 0:
                    lines.append('0 ' + ' '.join(f'{v:.6f}' for v in coords))
            except Exception:
                continue

    return lines

def save_pair(image_path: Path, yolo_lines, prefix: str, counter: int):
    split = 'train' if random.random() < 0.85 else 'val'
    out_img_name = f'{prefix}_{counter:06d}{image_path.suffix.lower()}'
    out_lbl_name = f'{prefix}_{counter:06d}.txt'

    shutil.copy(image_path, DATASET_ROOT / 'images' / split / out_img_name)
    (DATASET_ROOT / 'labels' / split / out_lbl_name).write_text('\n'.join(yolo_lines))

# Counters
image_counter = 0
added_rgbd = 0
added_voc = 0
added_extra = 0

# ── A) PothRGBD (YOLO txt) ───────────────────────────────────────────────────
print('Processing PothRGBD...')
poth_root = Path(pothrgbd_path)
poth_img_index = build_image_index(poth_root, skip_depth=True)

poth_label_files = [p for p in poth_root.rglob('*.txt') if 'label' in p.parent.name.lower()]
if not poth_label_files:
    poth_label_files = list(poth_root.rglob('*.txt'))

for lbl in poth_label_files:
    if lbl.name.lower().startswith('readme'):
        continue

    candidates = poth_img_index.get(lbl.stem, [])
    if not candidates:
        continue
    img_path = candidates[0]

    yolo_lines = parse_yolo_label_file(lbl)
    if not yolo_lines:
        continue

    save_pair(img_path, yolo_lines, prefix='rgbd', counter=image_counter)
    image_counter += 1
    added_rgbd += 1

print(f'  ✅ Added {added_rgbd} images from PothRGBD')

# ── B) Annotated Potholes (Pascal VOC XML) ──────────────────────────────────
print('\nProcessing Annotated Potholes (VOC XML)...')
ann_root = Path(annotated_path)
ann_img_index = build_image_index(ann_root, skip_depth=False)

for xml_path in ann_root.rglob('*.xml'):
    try:
        root = ET.parse(str(xml_path)).getroot()
        size_el = root.find('size')
        if size_el is None:
            continue
        img_w = float(size_el.find('width').text)
        img_h = float(size_el.find('height').text)

        candidates = ann_img_index.get(xml_path.stem, [])
        if not candidates:
            continue
        img_path = candidates[0]

        yolo_lines = []
        for obj in root.findall('object'):
            bb = obj.find('bndbox')
            if bb is None:
                continue
            xmin = max(0.0, float(bb.find('xmin').text))
            ymin = max(0.0, float(bb.find('ymin').text))
            xmax = min(img_w, float(bb.find('xmax').text))
            ymax = min(img_h, float(bb.find('ymax').text))

            w = (xmax - xmin) / img_w
            h = (ymax - ymin) / img_h
            if w <= 0 or h <= 0:
                continue

            xc = (xmin + xmax) / 2 / img_w
            yc = (ymin + ymax) / 2 / img_h
            yolo_lines.append(bbox_to_polygon_label(xc, yc, w, h))

        if not yolo_lines:
            continue

        save_pair(img_path, yolo_lines, prefix='voc', counter=image_counter)
        image_counter += 1
        added_voc += 1
    except Exception:
        continue

print(f'  ✅ Added {added_voc} images from Annotated Potholes')

# ── C) Optional uploaded ZIP datasets ────────────────────────────────────────
EXTRA_ZIP_DIR = Path('/content/extra_datasets')

if EXTRA_ZIP_DIR.exists() and any(EXTRA_ZIP_DIR.iterdir()):
    print(f'\nProcessing uploaded ZIP datasets from {EXTRA_ZIP_DIR} ...')

    def ingest_extra_dir(base_dir: Path, prefix: str, start_counter: int):
        local_added = 0
        counter = start_counter

        img_index = build_image_index(base_dir)

        # 1) YOLO txt labels
        yolo_label_files = [p for p in base_dir.rglob('*.txt') if 'label' in p.parent.name.lower()]
        if not yolo_label_files:
            yolo_label_files = list(base_dir.rglob('*.txt'))

        for lbl in yolo_label_files:
            if lbl.name.lower().startswith('readme'):
                continue
            candidates = img_index.get(lbl.stem, [])
            if not candidates:
                continue
            yolo_lines = parse_yolo_label_file(lbl)
            if not yolo_lines:
                continue
            save_pair(candidates[0], yolo_lines, prefix=prefix, counter=counter)
            counter += 1
            local_added += 1

        # 2) Pascal VOC XML
        for xml_path in base_dir.rglob('*.xml'):
            try:
                root = ET.parse(str(xml_path)).getroot()
                size_el = root.find('size')
                if size_el is None:
                    continue
                img_w = float(size_el.find('width').text)
                img_h = float(size_el.find('height').text)

                candidates = img_index.get(xml_path.stem, [])
                if not candidates:
                    continue

                yolo_lines = []
                for obj in root.findall('object'):
                    bb = obj.find('bndbox')
                    if bb is None:
                        continue
                    xmin = max(0.0, float(bb.find('xmin').text))
                    ymin = max(0.0, float(bb.find('ymin').text))
                    xmax = min(img_w, float(bb.find('xmax').text))
                    ymax = min(img_h, float(bb.find('ymax').text))

                    w = (xmax - xmin) / img_w
                    h = (ymax - ymin) / img_h
                    if w <= 0 or h <= 0:
                        continue

                    xc = (xmin + xmax) / 2 / img_w
                    yc = (ymin + ymax) / 2 / img_h
                    yolo_lines.append(bbox_to_polygon_label(xc, yc, w, h))

                if not yolo_lines:
                    continue

                save_pair(candidates[0], yolo_lines, prefix=prefix, counter=counter)
                counter += 1
                local_added += 1
            except Exception:
                continue

        # 3) COCO JSON
        for jf in base_dir.rglob('*.json'):
            try:
                coco = json.loads(jf.read_text())
            except Exception:
                continue
            if 'images' not in coco:
                continue

            ann_by_img = {}
            for ann in coco.get('annotations', []):
                ann_by_img.setdefault(ann['image_id'], []).append(ann)

            for img_info in coco.get('images', []):
                img_name = Path(img_info.get('file_name', '')).stem
                candidates = img_index.get(img_name, [])
                if not candidates:
                    continue

                img_w = float(img_info.get('width', 0) or 0)
                img_h = float(img_info.get('height', 0) or 0)
                if img_w <= 0 or img_h <= 0:
                    continue

                anns = ann_by_img.get(img_info.get('id'), [])
                if not anns:
                    continue

                yolo_lines = []
                for ann in anns:
                    seg = ann.get('segmentation', [])
                    if seg and isinstance(seg, list) and len(seg) and isinstance(seg[0], list):
                        poly = seg[0]
                        if len(poly) >= 6 and len(poly) % 2 == 0:
                            norm = [poly[i] / img_w if i % 2 == 0 else poly[i] / img_h for i in range(len(poly))]
                            norm = normalize_polygon(norm)
                            yolo_lines.append('0 ' + ' '.join(f'{v:.6f}' for v in norm))
                    elif 'bbox' in ann:
                        x, y, w, h = ann['bbox']
                        xc = (x + w / 2) / img_w
                        yc = (y + h / 2) / img_h
                        yolo_lines.append(bbox_to_polygon_label(xc, yc, w / img_w, h / img_h))

                if not yolo_lines:
                    continue

                save_pair(candidates[0], yolo_lines, prefix=prefix, counter=counter)
                counter += 1
                local_added += 1

        return counter, local_added

    for subdir in sorted(EXTRA_ZIP_DIR.iterdir()):
        if subdir.is_dir():
            image_counter, local_added = ingest_extra_dir(subdir, prefix=subdir.name[:8], start_counter=image_counter)
            added_extra += local_added
            print(f'  processed: {subdir.name} (+{local_added})')

    print(f'  ✅ Added {added_extra} images from uploaded ZIP datasets')
else:
    print('\nNo uploaded ZIPs found — using core datasets only.')

# ── Final sanity checks ──────────────────────────────────────────────────────
train_imgs = list((DATASET_ROOT / 'images' / 'train').iterdir())
val_imgs = list((DATASET_ROOT / 'images' / 'val').iterdir())

# Guarantee at least 1 validation sample if possible
if len(val_imgs) == 0 and len(train_imgs) > 1:
    src_img = train_imgs[0]
    src_lbl = (DATASET_ROOT / 'labels' / 'train' / f'{src_img.stem}.txt')
    dst_img = DATASET_ROOT / 'images' / 'val' / src_img.name
    dst_lbl = DATASET_ROOT / 'labels' / 'val' / src_lbl.name
    shutil.move(src_img, dst_img)
    if src_lbl.exists():
        shutil.move(src_lbl, dst_lbl)
    val_imgs = list((DATASET_ROOT / 'images' / 'val').iterdir())
    train_imgs = list((DATASET_ROOT / 'images' / 'train').iterdir())

print('\n📊 Final dataset summary')
print(f'  Total copied: {image_counter}')
print(f'  Train images: {len(train_imgs)}')
print(f'  Val images:   {len(val_imgs)}')
print(f'  RGBD added:   {added_rgbd}')
print(f'  VOC added:    {added_voc}')
print(f'  Extra added:  {added_extra}')

if len(train_imgs) == 0:
    raise RuntimeError('No training images were created. Stop and inspect dataset paths/output above.')


In [ ]:
# ── 9) Write data.yaml ──────────────────────────────────────────────────────
import yaml

data_yaml = {
    'path': str(DATASET_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['pothole'],
}

yaml_path = DATASET_ROOT / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(f'✅ data.yaml written: {yaml_path}')
print(yaml_path.read_text())

In [ ]:
# ── 10) Preview sample training images ───────────────────────────────────────
import cv2
import random
import matplotlib.pyplot as plt

train_imgs = list((DATASET_ROOT / 'images' / 'train').iterdir())
val_imgs = list((DATASET_ROOT / 'images' / 'val').iterdir())

print(f'Train images: {len(train_imgs)}')
print(f'Val images:   {len(val_imgs)}')

if len(train_imgs) == 0:
    raise RuntimeError('No train images found. Re-run cell 8 (dataset build).')

sample = random.sample(train_imgs, min(6, len(train_imgs)))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, img_path in zip(axes.flatten(), sample):
    img = cv2.imread(str(img_path))
    if img is None:
        ax.set_title(f'Unreadable: {img_path.name[:20]}')
        ax.axis('off')
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_path.name[:30])
    ax.axis('off')

for ax in axes.flatten()[len(sample):]:
    ax.axis('off')

plt.suptitle(f'Sample training images ({len(train_imgs)} total)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 11) Train YOLOv8x-seg (OOM-safe) ─────────────────────────────────────────
from ultralytics import YOLO
import torch, gc

train_count = len(list((DATASET_ROOT / 'images' / 'train').iterdir()))
val_count = len(list((DATASET_ROOT / 'images' / 'val').iterdir()))
print(f'Train images: {train_count}, Val images: {val_count}')

if train_count == 0:
    raise RuntimeError('Training aborted: dataset is empty. Re-run cell 8.')
if val_count == 0:
    raise RuntimeError('Training aborted: validation split is empty. Re-run cell 8.')

# T4-safe retry plan (largest -> smallest)
# If CUDA OOM/CUBLAS alloc error occurs, retry with lighter settings.
retry_plan = [
    {'imgsz': 896, 'batch': 4, 'workers': 2},
    {'imgsz': 768, 'batch': 2, 'workers': 1},
    {'imgsz': 640, 'batch': 2, 'workers': 1},
]

results = None
last_err = None

for i, cfg in enumerate(retry_plan, start=1):
    print(f"\n🚀 Training attempt {i}/{len(retry_plan)} with imgsz={cfg['imgsz']}, batch={cfg['batch']}, workers={cfg['workers']}")

    # Free GPU memory between retries
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = YOLO('yolov8x-seg.pt')

    try:
        results = model.train(
            data=str(yaml_path),
            epochs=80,
            imgsz=cfg['imgsz'],
            batch=cfg['batch'],
            device=0,
            workers=cfg['workers'],
            project='/content/runs',
            name='pothole-yolov8x-seg',
            exist_ok=True,
            hsv_h=0.015,
            hsv_s=0.7,
            hsv_v=0.4,
            degrees=10.0,
            translate=0.1,
            scale=0.5,
            flipud=0.3,
            fliplr=0.5,
            mosaic=1.0,
            mixup=0.1,
            copy_paste=0.2,
            patience=20,
            optimizer='AdamW',
            lr0=0.001,
            lrf=0.01,
            weight_decay=0.0005,
            warmup_epochs=3,
            cos_lr=True,
            conf=0.55,
            iou=0.45,
            save=True,
            save_period=10,
        )
        print('\n✅ Training complete!')
        print(f"Best weights: {results.save_dir}/weights/best.pt")
        break

    except RuntimeError as e:
        last_err = e
        msg = str(e).lower()
        if ('out of memory' in msg) or ('cublas_status_alloc_failed' in msg) or ('cuda error' in msg):
            print(f"⚠️ OOM on attempt {i}: {e}")
            if i < len(retry_plan):
                print('↪️ Retrying with lower memory settings...')
                continue
        raise

if results is None:
    raise RuntimeError(f'All training attempts failed. Last error: {last_err}')


In [ ]:
# ── 12) Validate final model ────────────────────────────────────────────────
from ultralytics import YOLO
from pathlib import Path

best_weights = Path('/content/runs/pothole-yolov8x-seg/weights/best.pt')
if not best_weights.exists():
    raise FileNotFoundError(f'Best weights not found: {best_weights}')

model = YOLO(str(best_weights))
metrics = model.val(data=str(yaml_path), imgsz=1024, conf=0.55, iou=0.45)

print('\n📊 Validation Results:')
print(f'  mAP50:       {metrics.seg.map50:.4f}')
print(f'  mAP50-95:    {metrics.seg.map:.4f}')
print(f'  Precision:   {metrics.seg.mp:.4f}')
print(f'  Recall:      {metrics.seg.mr:.4f}')

In [ ]:
# ── 13) Save best weights to Google Drive + download ───────────────────────
import shutil
from pathlib import Path
from google.colab import files

best_src = Path('/content/runs/pothole-yolov8x-seg/weights/best.pt')
if not best_src.exists():
    raise FileNotFoundError(f'No best.pt found at {best_src}')

best_dst = Path(SAVE_DIR) / 'yolov8x-seg-pothole.pt'
shutil.copy(best_src, best_dst)

print(f'✅ Saved to Drive: {best_dst}')
print("📦 Deploy as: backend/models/yolov8x-seg-pothole.pt")

files.download(str(best_src))